# Inspect and reproduce the article figures

The committed data were generated from the complete article experiment, including $n=1000$ and $2000$. This notebook reads those data; it does not recompute the expensive assignments by default.

Figure 1: discrete support cycles. Figure 2: cosine perturbations. Figure 3: radial annulus gap. Figure 4: empirical annulus assignments on the same article model. Read `figures/README.md` for the mathematical setup, seeds, grids and repetition protocol.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "iot_gap").is_dir())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
from iot_gap import (cosine_map, annulus_map, sample_pairs, sample_independent_marginals,
                     optimal_assignment, gap_curve, sampling_gap_curve)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
US = np.linspace(0, 1, 51)

In [ ]:
import json
DATA = ROOT / "figures" / "data"
files = sorted(DATA.glob("*.npz"))
assert files, "Run python figures/reproduce.py first, or use the committed data."
print(f"{len(files)} data archives available")
for path in files[:6]:
    with np.load(path, allow_pickle=False) as data:
        print(path.name, {key: data[key].shape for key in data.files})
for path in sorted(DATA.glob("*.json")):
    print("Metadata:", path.name)

## Raw arrays and normalization
Each gap archive stores exact evaluated grid nodes and loss values. Both source and target arrays are included so that values can be checked independently. The long CSV exports allow analysis without Python-specific serialization. Standard-deviation bands in the figures summarize repeated samples, with denominator equal to the number of repetitions; they are not confidence intervals. The two large cosine curves are single realizations.

Select any data archive below to inspect. The cell deliberately tolerates extra metadata keys while showing the shapes and numerical ranges.

In [ ]:
selected = files[0]
with np.load(selected, allow_pickle=False) as archive:
    for key in archive.files:
        value = archive[key]
        if np.issubdtype(value.dtype, np.number):
            print(key, "shape", value.shape, "range", (value.min(), value.max()))
        else:
            print(key, value)
print("Selected:", selected.name)

## View the exact publication assets
The SVG files below contain the same plotted data as the vector PDFs used in the manuscript. Use the PDF files for publication; PNG companions are provided for previews.

In [ ]:
from IPython.display import SVG, display
for path in sorted((ROOT / "figures" / "generated").glob("*.svg")):
    print(path.name)
    display(SVG(filename=str(path)))

## Regenerate rendering from cached values
This runs the plotting stage only and does not rerun the large assignment problems. Any changes to fonts, line widths or colors can be explored in the figure script and rendered again.

In [ ]:
import subprocess
# Uncomment to regenerate all vector and preview assets from committed raw data.
# subprocess.run([sys.executable, str(ROOT / "figures" / "reproduce.py"), "--render-only"], cwd=ROOT, check=True)
print("Rendering command: python figures/reproduce.py --render-only")

## Recompute from samples
Run `python figures/reproduce.py` from the repository root to compute missing cached experiments and render. To recompute one existing experiment, remove its `.npz` cache from `figures/data/` and rerun. To recompute everything, move `figures/data/` aside first. Full generation is substantially more expensive than these notebooks because exact dense assignments are evaluated at every article grid point, repeatedly up to n=2000. Check the generated manifest for the parameter settings, software versions and file checksums.

The browser experiments under `docs/` use smaller point counts and a different random generator. They implement the same mathematical experiments; their individual random clouds are not the publication samples.